In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import xgboost as xgb
from tqdm import tqdm
from pathlib import Path
from matplotlib import pyplot as plt
from importlib import import_module
from maw.training.metrics import compute_metrics
from maw.utils.loaders import load_features

In [ ]:
# Model Params
MODEL_STEM = "SENTINEL-XGBOOST-BESTPARAMS-ALL-FEATURES"   # stem for saved models
FEATURE_TYPE = "all-features"                             # features used during training
IMAGERY_TYPE = "sentinel"                                 # embeddings source imagery, if used
MODEL_CONFIG_NAME = 'maw.configs.sentinel_xgboost'  # config name from package root

In [ ]:
# Paths
HOME_DIR = Path.home() / "Projects/mapping-asia-wealth/"
MODEL_DIR = HOME_DIR / "data-clean/models/xgboost/"
PREDICTIONS_OUTDIR = HOME_DIR / "data-clean/outputs/predictions/"

## 5-fold Spatial Cross-Validation
We divide the dataset into five geographically independent folds—each a set of non-overlapping observation clusters—stratified by country to maintain balance. Clusters are assigned to folds via a greedy algorithm (i.e. iteratively placing the next largest cluster into the fold with the fewest total observations) to minimize size imbalance and avoid spatial leakage. We then perform 5-fold cross-validation (CV): in each iteration, one fold serves as the held-out (validation) set, and the model is trained on the other four folds. After each round, we generate out-of-fold predictions on the held-out fold.

To evaluate overall performance, we pool all out-of-fold predictions across the five iterations and compute our metrics (e.g., R^2, RMSE, MAE) on this aggregate. This ensures every observation is validated exactly once, yielding an unbiased estimate of generalization across countries.


In [ ]:
# Import configs
mod = import_module(MODEL_CONFIG_NAME)

In [ ]:
PATHS_DICT = {
    "labels-csv": mod.LABELS_PATH,
    "spatial-features-parquet": mod.COVARIATES_PATH,
    "embeddings-parquet": mod.EMBEDDINGS_PATH,
    # "aef-embeddings-parquet": mod.AEF_EMBEDDINGS_PATH,
    }

In [ ]:
LOADER_CONFIG = {
    "target_col": "wealth_index",
    "id_col": "patch_id",
    "fold_col": "fold_kmeans",
    "feature_type": FEATURE_TYPE,
    "paths_dict": PATHS_DICT,
    "as_numpy": False,
}

In [ ]:
# Evaluate on test set
X_test, y_test, meta = load_features(folds=[2, 3, 4, 5], return_meta=True, embed_backbone_heldoutset=1, **LOADER_CONFIG)
y_test = y_test.to_numpy()


In [ ]:
fold_stats = {}
outputs_list = []

for test_fold in tqdm(range(1, 6), desc="Evaluating", total=5):

    trained_model = xgb.XGBRegressor()
    trained_model.load_model(MODEL_DIR / f"{MODEL_STEM}-SPLIT{test_fold}.json")

    # Evaluate on test set
    X_test, y_test, meta = load_features(folds=[test_fold], return_meta=True, embed_backbone_heldoutset=test_fold, **LOADER_CONFIG)
    y_test = y_test.to_numpy()
    y_pred = trained_model.predict(X_test, validate_features=True)

    outputs_df = pd.DataFrame({"labels": y_test, "outputs": y_pred})
    outputs_df = pd.concat([outputs_df, meta], axis=1)
    test_mae, test_rmse, test_r2 = compute_metrics(y_true=y_test, y_pred=y_pred)

    print(f"Test RMSE: {test_rmse}")
    print(f"Test MAE: {test_mae}")
    print(f"Test R2: {test_r2}\n")

    outputs_list.append(outputs_df)
    fold_stats[f"test_fold{test_fold}"] = {"test_mae": test_mae, "test_rmse": test_rmse, "test_r2": test_r2}
    del trained_model

In [ ]:
# Save predictions and labels
fold_outputs = pd.concat(outputs_list)
fold_outputs.to_csv(PREDICTIONS_OUTDIR / f"xgboost-{FEATURE_TYPE.lower()}-{IMAGERY_TYPE.lower()}-test-set-predictions.csv", index=False)

In [ ]:
# Compute validation metrics for countries individually
country_codes = list(fold_outputs["country_code"].unique())
country_outputs_list = []
for cc in country_codes:
    if cc in ["KH", "KY"]:
        continue
    subset_df = fold_outputs.query(f"country_code == '{cc}'").reset_index()
    val_mae, val_rmse, val_r2 = compute_metrics(y_true=np.array(subset_df["labels"]), y_pred=np.array(subset_df["outputs"]))
    country_outputs_list.append({"country_code": cc, "mae": val_mae, "rmse": val_rmse, "r2": val_r2})

country_metrics = pd.DataFrame(country_outputs_list)
country_metrics

In [ ]:
fold_outputs

In [ ]:
# ------ Plot Validation Performance on DHS AWI, for All Countries (Separately) ------

# Create a FacetGrid with one facet per country
g = sns.FacetGrid(fold_outputs, col="country_code", hue="rural", height=4, aspect=1, col_wrap=4)
g.map_dataframe(sns.scatterplot, x="labels", y="outputs", s=50, alpha=0.5, edgecolor="w")

# Plot diagonal line corresponding to the perfect prediction
for ax in g.axes.flatten():
    xlims = ax.get_xlim()
    ylims = ax.get_ylim()
    min_val = min(xlims[0], ylims[0])
    max_val = max(xlims[1], ylims[1])

    ax.plot([min_val, max_val], [min_val, max_val],
            linestyle="--", color="red", label="Perfect Prediction")

    ax.set_xlim(xlims)
    ax.set_ylim(ylims)

g.add_legend()
g.set_axis_labels("Actual", "Predicted")
g.fig.suptitle("Performance on All-Countries, XGBoost w. Resnet50 Image Embeddings", y=1.05)

plt.show()